# 3 Who can the head serve?

A cascade sends the easy cases to a cheap model and escalates the rest. To build one you
first need to know what "easy" means *for this model on this data* — which classes the cheap
head can handle, and which it cannot.

SCOTUS makes the question sharp. Thirteen issue areas, and the smallest has 35 documents in
the entire dataset against 1400 for the largest. A head trained on that will predict the big
classes and ignore the small ones, and it is tempting to read that as "the rare classes are
impossible, send them to an LLM".

That reading is only valid if the head was given a fair chance. This notebook gives it one —
class balancing, noise augmentation, and a search over the architecture — and then measures
what is left. Whatever survives a properly tuned head is the real case for escalation.

In [ ]:
import os
import time
import warnings
from pathlib import Path

import numpy as np
import torch

os.environ.setdefault("RAY_ENABLE_UV_RUN_RUNTIME_ENV", "0")
warnings.simplefilter("ignore", UserWarning)

from scotus import AREAS, CACHE_REPO
from vectormesh import VectorCache
from vectormesh.data.vectorizers import detect_device

K = len(AREAS)
device = detect_device()
ARTEFACTS = Path("../../artefacts").resolve()

traincache = VectorCache.from_hub(CACHE_REPO, split="train")
testcache = VectorCache.from_hub(CACHE_REPO, split="test")
print(f"{len(traincache.dataset)} train / {len(testcache.dataset)} test opinions, {K} issue areas")

## 3.1 The shape of the problem

The cache is the one published in `build_scotus_cache.py`: every opinion cut to 24.000
characters, encoded by `bge-large-en-v1.5` in 512-token chunks. Each row is a `(chunks, 1024)`
matrix, not a vector — folding that back to one vector is a choice, and lesson 4 showed it is
a choice worth searching over rather than defaulting to a mean.

In [ ]:
import matplotlib.pyplot as plt

y_train = np.asarray([int(v) for v in traincache.dataset["label"]])
y_test = np.asarray([int(v) for v in testcache.dataset["label"]])
counts = np.bincount(y_train, minlength=K)
order = np.argsort(-counts)

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.bar(range(K), counts[order])
ax.set_xticks(range(K))
ax.set_xticklabels([AREAS[i] for i in order], rotation=45, ha="right", fontsize=8)
ax.set_ylabel("training documents")
ax.set_title(f"{counts.max()}x between the largest and smallest issue area")
plt.tight_layout()
plt.show()

print(f"{'issue area':24s} {'train':>6s} {'test':>5s}")
for i in order:
    print(f"{AREAS[i]:24s} {counts[i]:6d} {int((y_test == i).sum()):5d}")

In [ ]:
meta = traincache.metadata["embed"]
chunks = {int(k): v for k, v in meta["chunk_sizes"].items()}
total = sum(chunks.values())

for cutoff in (4, 8, 16, 24, 32):
    kept = sum(v for k, v in chunks.items() if k <= cutoff)
    print(f"max_chunks={cutoff:3d}: {total - kept:5d} opinions truncated "
          f"({(total - kept) / total:5.1%})")

## 3.2 Two things to stack against the imbalance

They compose — that is the point. Balancing decides *how often* the model sees a rare class,
noise decides *what it sees* when it does. Both are axes in the search, so trials exist with
either, both, or neither.

**Balanced sampling.** A `WeightedRandomSampler` draws each class equally often, so a class
with 35 documents is seen as often as one with 1400. The model stops being able to win by
ignoring it.

**Noise augmentation.** Sampling a 35-document class 40 times an epoch shows the model the
same 35 points over and over. `GaussianNoise` jitters each draw, so it sees a neighbourhood
instead of a point. It is a component, so it sits in the pipeline and turns itself off at
eval — `sigma` becomes a hyperparameter like any other.

Neither is free: balancing spends capacity on classes that are rare for a reason, and too
much noise destroys the signal. Neither belongs in an assumption.

In [ ]:
from torch.utils.data import DataLoader, WeightedRandomSampler
from vectormesh.components import FixedPadding
from vectormesh.data import Collate, OneHot

onehot = OneHot(num_classes=K, label_col="label", target_col="onehot")
train_oh = traincache.map(onehot)
test_oh = testcache.map(onehot)

split = train_oh.train_test_split(test_size=800, seed=42)
tune_train_ds, tune_valid_ds = split["train"], split["test"]
print(f"tune-train {len(tune_train_ds)} / tune-valid {len(tune_valid_ds)} / "
      f"held-out test {len(test_oh)} (untouched until 3.6)")


def make_loader(ds, max_chunks, balanced=False, shuffle=True, batch_size=32):
    collate = Collate(embedding_col="embed", target_col="onehot",
                      padder=FixedPadding(max_chunks=max_chunks))
    if balanced:
        labels = np.asarray([int(v) for v in ds["label"]])
        freq = np.bincount(labels, minlength=K)
        weights = 1.0 / np.maximum(freq[labels], 1)
        sampler = WeightedRandomSampler(weights.tolist(), len(labels), replacement=True)
        return DataLoader(ds, batch_size=batch_size, sampler=sampler,
                          collate_fn=collate, num_workers=0)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                      collate_fn=collate, num_workers=0)

## 3.3 The pipeline

`GaussianNoise` first, on the raw chunk embeddings. Then a projection to a working width,
optional transformer blocks that let chunks talk to each other, an aggregator that collapses
the chunk axis, dropout, and a classifier head.

`hidden_dim` is searched from 32 to 128. Whether that is a *bottleneck* depends on where the
search lands relative to the 1024 dims coming in — it is a width, and the search decides
whether squeezing helps.

In [ ]:
from vectormesh.components import (
    AttentionAggregator, GaussianNoise, MaskedMeanAggregator, MeanAggregator,
    NeuralNet, Projection, RNNAggregator, Serial, TransformerBlock,
)

D_IN = meta["hidden_size"]

AGGREGATORS = {
    "mean": lambda d: MeanAggregator(),
    "masked_mean": lambda d: MaskedMeanAggregator(),
    "attention": lambda d: AttentionAggregator(hidden_size=d),
    "rnn": lambda d: RNNAggregator(hidden_size=d),
}


def build_pipeline(config):
    parts = [GaussianNoise(sigma=config["sigma"], relative=True),
             Projection(hidden_size=D_IN, out_size=config["hidden_dim"])]
    for _ in range(config["n_transformer_blocks"]):
        parts.append(TransformerBlock(hidden_size=config["hidden_dim"],
                                      num_heads=config["num_heads"]))
    parts.append(AGGREGATORS[config["aggregator"]](config["hidden_dim"]))
    parts.append(torch.nn.Dropout(config["dropout"]))
    parts.append(NeuralNet(hidden_size=config["hidden_dim"], out_size=K))
    return Serial(parts)

## 3.4 What to measure

Accuracy on this data is mostly a report on the two biggest classes: predict *Criminal
Procedure* and *Economic Activity* well and you are at 0.6 without ever touching the tail.

Macro recall averages recall over classes, so every issue area counts the same regardless of
size. A model that ignores six classes cannot score well on it — but it is also gameable:
predict a rare class generously and its recall rises while its precision collapses, and macro
recall never notices.

Macro F1 does notice, because it is the harmonic mean of the two. That makes it the metric
the winner is chosen on here; the other two are reported alongside so you can see what the
choice costs.

In [ ]:
def train_fn(config, *, train_ds, valid_ds, epochs):
    from pathlib import Path

    import torch
    from ray import tune
    from sklearn.metrics import f1_score, recall_score
    from vectormesh.components.metrics import Accuracy
    from vectormesh.training import Trainer, TrainerSettings

    tr = make_loader(train_ds, config["max_chunks"], balanced=config["balanced"])
    va = make_loader(valid_ds, config["max_chunks"], shuffle=False)

    model = build_pipeline(config)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    def to_ray(epoch, train_loss, test_loss, metric_dict):
        model.eval()
        preds, gold = [], []
        with torch.no_grad():
            for X, y in va:
                preds += model(X).argmax(-1).tolist()
                gold += y.argmax(-1).tolist()
        tune.report({
            "accuracy": metric_dict["Accuracy"],
            "macro_recall": recall_score(gold, preds, average="macro", zero_division=0),
            "macro_f1": f1_score(gold, preds, average="macro", zero_division=0),
            "classes_predicted": len(set(preds)),
            "params": n_params,
        })

    settings = TrainerSettings(
        epochs=epochs,
        metrics=[Accuracy()],
        logdir=Path("ray_results/trials").resolve(),
        train_steps=len(tr),
        valid_steps=len(va),
        earlystop_kwargs=None,
    )
    Trainer(
        model=model,
        settings=settings,
        loss_fn=torch.nn.CrossEntropyLoss(),
        optimizer=torch.optim.Adam,
        traindataloader=tr,
        validdataloader=va,
        reporters=[to_ray],
        progress=False,
    ).loop()

## 3.5 One search over architecture *and* balancing

Searching the architecture with balancing fixed, or balancing with the architecture fixed,
answers a question nobody asked: the best aggregator for an unbalanced head need not be the
best one for a balanced head. They go in the same space.

In [ ]:
import pandas as pd
import ray
from notebooktester import param
from ray import tune
from ray.tune import JupyterNotebookReporter
from ray.tune.search.hyperopt import HyperOptSearch

SEARCH_SPACE = {
    "aggregator": tune.choice(["mean", "masked_mean", "attention", "rnn"]),
    "max_chunks": tune.choice([4, 8, 16, 24]),
    "hidden_dim": tune.qrandint(32, 128, 16),
    "n_transformer_blocks": tune.choice([0, 1, 2]),
    "num_heads": tune.choice([1, 2, 4]),
    "balanced": tune.choice([True, False]),
    "sigma": tune.uniform(0.0, 0.15),
    "dropout": tune.uniform(0.0, 0.4),
}
EPOCHS, N_TRIALS = param(10, test=1), param(40, test=1)
TUNE_DIR = ARTEFACTS / "scotus_head_tune"

t0 = time.time()
tuner = tune.Tuner(
    tune.with_parameters(train_fn, train_ds=tune_train_ds, valid_ds=tune_valid_ds,
                         epochs=EPOCHS),
    param_space=SEARCH_SPACE,
    tune_config=tune.TuneConfig(metric="macro_f1", mode="max",
                                search_alg=HyperOptSearch(metric="macro_f1", mode="max"),
                                num_samples=N_TRIALS),
    run_config=tune.RunConfig(
        storage_path=str(TUNE_DIR),
        progress_reporter=JupyterNotebookReporter(
            metric_columns=["accuracy", "macro_recall", "macro_f1", "classes_predicted"],
            max_report_frequency=20),
    ),
)
results = tuner.fit()
print(f"\n{N_TRIALS} trials in {time.time() - t0:.0f}s")

In [ ]:
df = results.get_dataframe()
by_acc = df.sort_values("accuracy", ascending=False).iloc[0]
by_rec = df.sort_values("macro_recall", ascending=False).iloc[0]
by_f1 = df.sort_values("macro_f1", ascending=False).iloc[0]

print(f"{'':22s} {'accuracy':>9s} {'macro_recall':>13s} {'macro_f1':>9s} {'classes':>8s}")
for name, row in (("best by accuracy", by_acc), ("best by macro recall", by_rec),
                  ("best by macro f1", by_f1)):
    print(f"{name:22s} {row['accuracy']:9.3f} {row['macro_recall']:13.3f} "
          f"{row['macro_f1']:9.3f} {int(row['classes_predicted']):8d}")
    print(f"  {({k.replace('config/', ''): row[k] for k in df.columns if k.startswith('config/')})}")

### Which axis actually moved the number?

A single winning trial is one draw. Averaging over every trial that shares a setting is a
much better estimate of whether that setting helps.

In [ ]:
cols = ["accuracy", "macro_recall", "macro_f1", "classes_predicted"]
print(df.groupby("config/balanced")[cols].mean().round(3), "\n")
print(df.groupby("config/aggregator")[cols].mean().round(3), "\n")

# sigma and dropout are continuous now, so bin them before averaging
for axis in ("config/sigma", "config/dropout"):
    binned = pd.qcut(df[axis], 3, duplicates="drop")
    print(df.groupby(binned, observed=True)[cols].mean().round(3), "\n")

## 3.6 Retrain the winner honestly

The search used a short, equal budget per trial — enough to rank configurations, not to
report a number. The winner is retrained on the full training split with early stopping, and
only now does the test set get touched.

In [ ]:
from sklearn.metrics import recall_score
from vectormesh.components.metrics import Accuracy
from vectormesh.training import Trainer, TrainerSettings

FINAL_EPOCHS, PATIENCE = param(60, test=1), 8


def fit_final(config, seed=0):
    torch.manual_seed(seed)
    tr = make_loader(train_oh, config["max_chunks"], balanced=config["balanced"])
    va = make_loader(test_oh, config["max_chunks"], shuffle=False)
    model = build_pipeline(config)
    settings = TrainerSettings(
        epochs=FINAL_EPOCHS,
        metrics=[Accuracy()],
        logdir=Path("ray_results/final").resolve(),
        train_steps=len(tr),
        valid_steps=len(va),
        earlystop_kwargs={"patience": PATIENCE, "save": False, "verbose": False},
    )
    trainer = Trainer(model=model, settings=settings, loss_fn=torch.nn.CrossEntropyLoss(),
                      optimizer=torch.optim.Adam, traindataloader=tr, validdataloader=va,
                      device=str(device))
    trainer.loop()
    return model, trainer


def evaluate(model, config):
    loader = make_loader(test_oh, config["max_chunks"], shuffle=False)
    model.eval()
    preds, gold = [], []
    with torch.no_grad():
        for X, y in loader:
            preds += model(X.to(device)).argmax(-1).cpu().tolist()
            gold += y.argmax(-1).tolist()
    return np.asarray(preds), np.asarray(gold)


def as_config(row):
    # A ray dataframe row back to a plain config: its values are numpy scalars,
    # which torch and json both refuse in their own ways.
    return {k.replace("config/", ""): (v.item() if hasattr(v, "item") else v)
            for k, v in row.items() if k.startswith("config/")}


winner = as_config(by_f1)
print("retraining:", winner)

model, trainer = fit_final(winner)
pred, gold = evaluate(model, winner)
print(f"\nheld-out test: accuracy {float((pred == gold).mean()):.3f}, "
      f"macro recall {recall_score(gold, pred, average='macro', zero_division=0):.3f}")

## 3.7 Per-class recall is the routing decision

This table is what the router is built on. A class the head serves well should stay with the
head; a class it cannot serve is where an expensive model might earn its cost.

Note what does *not* work as a rule: sorting by training count. If recall tracked size, you
could route on the counts alone and never train anything.

In [ ]:
recall = np.array([float((pred[gold == k] == k).mean()) if (gold == k).sum() else np.nan
                   for k in range(K)])
support = np.array([int((gold == k).sum()) for k in range(K)])

rank = np.argsort(-recall)
print(f"{'issue area':24s} {'n_train':>8s} {'n_test':>7s} {'recall':>7s}")
for k in rank:
    print(f"{AREAS[k]:24s} {counts[k]:8d} {support[k]:7d} {recall[k]:7.3f}")

corr = np.corrcoef(counts, np.nan_to_num(recall))[0, 1]
print(f"\ncorrelation between training count and recall: {corr:+.2f}")

In [ ]:
TAU = 0.60

owned = [k for k in range(K) if recall[k] >= TAU]
tail = [k for k in range(K) if k not in owned]
share_owned = float(np.isin(gold, owned).mean())

print(f"tau = {TAU}")
print(f"owned by the head ({len(owned)}): {[AREAS[k] for k in owned]}")
print(f"escalate ({len(tail)}):          {[AREAS[k] for k in tail]}")
print(f"\n{share_owned:.1%} of test documents fall in an owned class — "
      f"the fraction a perfect router would keep cheap")

`TAU` is a knob, not a discovery: raise it and the head keeps fewer classes but serves them
better, lower it and more traffic stays cheap. The next notebook sweeps it against the cost
of an LLM call instead of fixing it by hand.

In [ ]:
import json

profile = {
    "config": winner,
    "tau": TAU,
    "owned": owned,
    "tail": tail,
    "recall": [None if np.isnan(r) else float(r) for r in recall],
    "n_train": counts.tolist(),
    "n_test": support.tolist(),
    "test_accuracy": float((pred == gold).mean()),
    "test_macro_recall": float(recall_score(gold, pred, average="macro", zero_division=0)),
}
out = Path("data/head_profile.json")
out.parent.mkdir(exist_ok=True)
out.write_text(json.dumps(profile, indent=1))
print(f"wrote {out}")

## 3.8 What this feeds

This notebook produced one half of the cascade: a head, and a measured statement of which
classes it serves. The system it belongs to is

    embedding -> router -> { head | LLM }

and the router is the missing half. Note what the head profile bought: a *label* for training
it. "Is this document in a class the head serves above tau" is a yes/no question answerable
from `head_profile.json` and the training labels alone — no LLM calls needed to build the
router, only to measure what escalation gains.

One structural warning for the next notebook. The head is a `Serial` pipeline and the router
will be another model over the same embeddings, but the cascade itself is **not** a vectormesh
pipeline: an LLM call is an HTTP request against text, not a differentiable module over
tensors, and it cannot sit inside `Serial` the way `MoE` holds its experts. The router is a
model; the routing is orchestration around it. Conflating the two is how people end up trying
to backpropagate through an API.

## Exercises

**E1 — is the winner actually better, or luckier?** The search ranked configurations on 800
validation documents. Retrain the top three configurations over three seeds each and compare
the spread within a configuration to the gap between them.

**E2 — the metric chooses the model.** Re-run 3.6 with the best-by-accuracy configuration
instead. How many classes does it serve above `TAU`, and what does that do to the fraction of
traffic that has to be escalated?

**E3 — the cheap alternative.** Logit adjustment subtracts the log prior from a trained
model's outputs at inference, needing no retraining at all. Apply it to the unbalanced head
and compare per-class recall against the balanced one. If it comes close, what did the
balanced training buy?

**E4 — when the head is wrong, how wrong is it?** For each escalated class, look at what the
head predicts instead. If the errors concentrate on one or two classes, a cheaper fix than an
LLM may exist.

**E5 — does macro F1 deserve to be the selection metric?** Retrain the best-by-macro-recall
configuration and compare per-class *precision* against the best-by-F1 one. Which classes does
generous recall cost you, and would the router notice?